# Fair Frames Presentation Demo

使用 `Fair frames presentation.pptx` 中的布局生成 PPT，验证内容与布局是否匹配。

In [4]:
import sys
sys.path.insert(0, '../src')

from pptx import Presentation
from template_manager import load_template, get_layout_mapping, print_layouts

template_file = '../templates/Fair frames presentation.pptx'
prs = load_template(template_file)

print('Available Slide Layouts in Fair frames presentation.pptx:')
print_layouts(prs)

Available Slide Layouts in Fair frames presentation.pptx:
Layout 0: Title
Layout 1: Agenda
Layout 2: Section Header 1
Layout 3: Section Header 2
Layout 4: Introduction
Layout 5: Section Header 3
Layout 6: Comparison
Layout 7: Two Content 2
Layout 8: Summary
Layout 9: Title Content and Table
Layout 10: Two Content
Layout 11: Table
Layout 12: Thank you


In [5]:
from input_parser import parse_input_text
from ppt_generator import generate_presentation

# 使用 Fair Frames 模板的布局名称替换布局名称
input_text = """
    # ChatPPT_Demo

    ## ChatPPT Demo [Title Only]

    ## 2024 业绩概述 [Title and Content]
    - 总收入增长15%
    - 市场份额扩大至30%

    ## 业绩图表 [Title and Picture 1]
    ![业绩图表](images/performance_chart.png)

    ## 新产品发布 [Title and 2 Column]
    - 产品A: 特色功能介绍
    - 产品B: 市场定位
    ![未来增长](images/forecast.png)
    """

layout_mapping = get_layout_mapping(prs)
print('Layout mapping:')
for name, idx in layout_mapping.items():
    print(f'  [{idx}] {name}')

Layout mapping:
  [0] Title
  [1] Agenda
  [2] Section Header 1
  [3] Section Header 2
  [4] Introduction
  [5] Section Header 3
  [6] Comparison
  [7] Two Content 2
  [8] Summary
  [9] Title Content and Table
  [10] Two Content
  [11] Table
  [12] Thank you


In [6]:
powerpoint_data, presentation_title = parse_input_text(input_text, layout_mapping)

print(f'Presentation title: {presentation_title}')
print(f'Number of slides: {len(powerpoint_data.slides)}')
print()

reverse_mapping = {v: k for k, v in layout_mapping.items()}
for i, slide in enumerate(powerpoint_data.slides):
    layout_name = reverse_mapping.get(slide.layout, 'Unknown')
    print(f'Slide {i+1}: Layout={slide.layout} ({layout_name}), Title="{slide.content.title}"')
    for bp in slide.content.bullet_points:
        print(f'         - {bp}')

Presentation title: ChatPPT_Demo
Number of slides: 4

Slide 1: Layout=1 (Agenda), Title="ChatPPT Demo"
Slide 2: Layout=1 (Agenda), Title="2024 业绩概述"
         - 总收入增长15%
         - 市场份额扩大至30%
Slide 3: Layout=1 (Agenda), Title="业绩图表"
Slide 4: Layout=1 (Agenda), Title="新产品发布"
         - 产品A: 特色功能介绍
         - 产品B: 市场定位


In [8]:
import os
os.makedirs('../outputs', exist_ok=True)

output_pptx = f'../outputs/{presentation_title}.pptx'
generate_presentation(powerpoint_data, template_file, output_pptx)
print(f'Done! File saved: {output_pptx}')

所有默认幻灯片已被移除。
Presentation saved to '../outputs/ChatPPT_Demo.pptx'
Done! File saved: ../outputs/ChatPPT_Demo.pptx


## 验证生成结果

读取输出文件，检查每张幻灯片的布局名称与内容是否与输入匹配。

In [9]:
result_prs = Presentation(output_pptx)
print(f'Generated slides: {len(result_prs.slides)}')
print()

all_ok = True
for i, slide in enumerate(result_prs.slides):
    layout_name = slide.slide_layout.name
    title_text = slide.shapes.title.text if slide.shapes.title else '(no title)'

    texts = []
    for shape in slide.shapes:
        if shape.has_text_frame and shape != slide.shapes.title:
            for para in shape.text_frame.paragraphs:
                t = para.text.strip()
                if t:
                    texts.append(t)

    expected_layout = powerpoint_data.slides[i]
    expected_layout_name = reverse_mapping.get(expected_layout.layout, 'Unknown')
    match = '✓' if layout_name == expected_layout_name else '✗ MISMATCH'
    if layout_name != expected_layout_name:
        all_ok = False

    print(f'--- Slide {i+1} [{match}] ---')
    print(f'  Layout  : {layout_name} (expected: {expected_layout_name})')
    print(f'  Title   : {title_text}')
    if texts:
        print(f'  Content :')
        for t in texts:
            print(f'    • {t}')
    print()

if all_ok:
    print('All slides: layout matches content ✓')
else:
    print('Some slides have layout mismatches ✗')

Generated slides: 4

--- Slide 1 [✓] ---
  Layout  : Agenda (expected: Agenda)
  Title   : ChatPPT Demo

--- Slide 2 [✓] ---
  Layout  : Agenda (expected: Agenda)
  Title   : 2024 业绩概述
  Content :
    • 总收入增长15%
    • 市场份额扩大至30%

--- Slide 3 [✓] ---
  Layout  : Agenda (expected: Agenda)
  Title   : 业绩图表

--- Slide 4 [✓] ---
  Layout  : Agenda (expected: Agenda)
  Title   : 新产品发布
  Content :
    • 产品A: 特色功能介绍
    • 产品B: 市场定位

All slides: layout matches content ✓
